In [2]:
import torch
from torch import nn
import torch.nn.functional as F

In [ ]:
EOS_TOKEN = 0
PAD_TOKEN = 1

batch_size = 32
vocab_size = 512
k = 4
alpha = 0.6
max_len = 20

log_probs = torch.zeros(batch_size, k)  #B, K

a = F.log_softmax(torch.randn(batch_size, vocab_size), dim=-1) #B, VOCAB
top_a, loc = torch.topk(a, k=k, dim=-1) #B, 4

active = torch.ones((batch_size, k), dtype=torch.bool)
active = active & ~(loc == EOS_TOKEN)

log_probs += top_a

# Each beam starts with its chosen vocab index
beams = loc.unsqueeze(-1)

for i in range(1, max_len):

    new_a = F.log_softmax(torch.randn(batch_size, k, vocab_size), dim=-1)
    
    #always select a finished beam 
    new_a[active == False] = float('-inf') 
    new_a[active == False, PAD_TOKEN] = 0.0
    new_a[active, PAD_TOKEN] = float('-inf')

    new_a += log_probs.unsqueeze(-1)
    flat_new_a = new_a.view(batch_size, -1)
    top2_vals, top2_indices = torch.topk(flat_new_a, k=k, dim=-1)

    log_probs = top2_vals
    # Map flat indices back to (beam, token) pairs
    beam_indices = top2_indices // vocab_size
    token_indices = top2_indices % vocab_size

    # Update beams
    batch_idx = torch.arange(batch_size).unsqueeze(1)
    beams = beams[batch_idx, beam_indices, :]
    beams = torch.cat((beams, token_indices.unsqueeze(-1)), dim=-1)
    
    active = active[batch_idx, beam_indices]
    just_finished = token_indices == EOS_TOKEN
    active = active & ~just_finished
    if not active.any():
        break

# Divide log_probs of each beam by (length - number of pad tokens) ** alpha
# beams: (batch_size, k, seq_len_so_far)
# log_probs: (batch_size, k)
lengths = beams.shape[-1]

num_pad = (beams == PAD_TOKEN).sum(dim=-1)

normalized_lengths = (lengths - num_pad).clamp(min=1)

log_probs = log_probs / (normalized_lengths.float() ** alpha)
print(beams)
    #check if any beam reached EOS token

tensor([[[479, 144, 313,  ..., 322,  30, 453],
         [479, 144, 313,  ..., 129,  15, 266],
         [479, 144, 313,  ..., 380, 230, 445],
         [479, 144, 313,  ..., 322,  30,  28]],

        [[372, 393,  56,  ..., 266, 394, 505],
         [372, 393,  56,  ..., 224, 407,  91],
         [372, 393,  56,  ..., 266, 394, 498],
         [372, 393,  56,  ..., 266, 394, 465]],

        [[356, 126, 119,  ..., 437, 113,  16],
         [356, 126, 119,  ..., 437, 113,  62],
         [356, 126, 119,  ..., 437, 113, 408],
         [356, 126, 119,  ..., 437, 113,  17]],

        ...,

        [[442, 360, 241,  ..., 103, 205, 427],
         [442, 360, 241,  ..., 103, 205, 422],
         [442, 360, 241,  ..., 103, 205, 238],
         [442, 360, 241,  ..., 103, 205, 130]],

        [[235, 507, 167,  ..., 338,  16,  33],
         [235, 507, 167,  ..., 338,  16,  79],
         [235, 507, 167,  ..., 338,  16, 229],
         [235, 507, 167,  ...,  43, 397, 225]],

        [[176, 338, 253,  ...,  24, 